<a href="https://colab.research.google.com/github/M-Santhoshkumar-2005/ITA0616-MACHINE-LEARNING-SLOT-A-/blob/main/Experiment_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd


# =====================================================================
# Embedded Dataset (Based on Unit III Outlook & Temperature Play Sport)
# Columns: [Outlook, Temperature, Play]
# =====================================================================
dataset = [
    ["Sunny", "Hot", "No"],
    ["Sunny", "Hot", "No"],
    ["Overcast", "Hot", "Yes"],
    ["Rain", "Mild", "Yes"],
    ["Rain", "Cool", "Yes"],
    ["Rain", "Cool", "No"],
    ["Overcast", "Cool", "Yes"],
    ["Sunny", "Mild", "No"],
    ["Sunny", "Cool", "Yes"],
    ["Rain", "Mild", "Yes"],
    ["Sunny", "Mild", "Yes"],
    ["Overcast", "Mild", "Yes"],
    ["Overcast", "Hot", "Yes"],
    ["Rain", "Mild", "No"],
    ["Sunny", "Hot", "No"],
    ["Rain", "Cool", "Yes"],
]

df = pd.DataFrame(dataset, columns=["Outlook", "Temperature", "Play"])
print("Dataset Preview:")
print(df.to_string(index=False))
print()


class CategoricalNaiveBayes:
    def __init__(self, alpha=1.0):
        # Laplace smoothing parameter (alpha=1.0 prevents zero-probability issues)
        self.alpha = alpha
        self.class_priors = {}
        self.feature_probs = {}
        self.classes = []
        self.feature_domains = {}

    def fit(self, X_df, y_series):
        n_samples = len(X_df)
        self.classes = sorted(y_series.unique())

        for col in X_df.columns:
            self.feature_domains[col] = sorted(X_df[col].unique())

        for c in self.classes:
            subset = X_df[y_series == c]
            n_c = len(subset)
            self.class_priors[c] = n_c / n_samples
            self.feature_probs[c] = {}

            for col in X_df.columns:
                v_k = len(self.feature_domains[col])
                value_counts = subset[col].value_counts().to_dict()
                col_probs = {}
                for val in self.feature_domains[col]:
                    count_val = value_counts.get(val, 0)
                    # Smoothed conditional probability P(X_i = val | Class = c)
                    col_probs[val] = (count_val + self.alpha) / (n_c + self.alpha * v_k)
                self.feature_probs[c][col] = col_probs

    def predict_proba_row(self, row):
        posteriors = {}
        for c in self.classes:
            prob = self.class_priors[c]
            for col in row.index:
                val = row[col]
                if val in self.feature_probs[c][col]:
                    prob *= self.feature_probs[c][col][val]
                else:
                    v_k = len(self.feature_domains[col])
                    n_c = int(round(self.class_priors[c] * len(row)))
                    prob *= self.alpha / (max(n_c, 1) + self.alpha * v_k)
            posteriors[c] = prob

        total = sum(posteriors.values())
        if total > 0:
            for c in posteriors:
                posteriors[c] /= total
        return posteriors

    def predict(self, X_df):
        preds = []
        probs_list = []
        for _, row in X_df.iterrows():
            probs = self.predict_proba_row(row)
            probs_list.append(probs)
            best_class = max(probs, key=probs.get)
            preds.append(best_class)
        return preds, probs_list


def compute_confusion_matrix(y_true, y_pred, labels):
    label_to_idx = {lbl: idx for idx, lbl in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=int)
    for actual, pred in zip(y_true, y_pred):
        if actual in label_to_idx and pred in label_to_idx:
            cm[label_to_idx[actual], label_to_idx[pred]] += 1
    return cm


# --- Train / Test Split ---
split_idx = 11
train_df = df.iloc[:split_idx].reset_index(drop=True)
test_df = df.iloc[split_idx:].reset_index(drop=True)

X_train, y_train = train_df[["Outlook", "Temperature"]], train_df["Play"]
X_test, y_test = test_df[["Outlook", "Temperature"]], test_df["Play"]

nb = CategoricalNaiveBayes(alpha=1.0)
nb.fit(X_train, y_train)

y_pred, test_probs = nb.predict(X_test)

# --- Display Test Predictions ---
print("=== Test Set Predictions ===")
for i in range(len(test_df)):
    row_info = X_test.iloc[i].to_dict()
    p_info = {k: round(v, 4) for k, v in test_probs[i].items()}
    print(
        f"Sample {i + 1} | {row_info} | Actual: {y_test.iloc[i]:3s} | "
        f"Predicted: {y_pred[i]:3s} | Probs: {p_info}"
    )

# --- Confusion Matrix and Accuracy ---
labels = ["Yes", "No"]
cm = compute_confusion_matrix(y_test.tolist(), y_pred, labels)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {lbl}" for lbl in labels],
    columns=[f"Pred {lbl}" for lbl in labels],
)

correct = np.trace(cm)
total = np.sum(cm)
accuracy = (correct / total) * 100 if total > 0 else 0.0

print("\n=== Confusion Matrix ===")
print(cm_df)
print(f"\nAccuracy: {accuracy:.2f}% ({correct}/{total})")

Dataset Preview:
 Outlook Temperature Play
   Sunny         Hot   No
   Sunny         Hot   No
Overcast         Hot  Yes
    Rain        Mild  Yes
    Rain        Cool  Yes
    Rain        Cool   No
Overcast        Cool  Yes
   Sunny        Mild   No
   Sunny        Cool  Yes
    Rain        Mild  Yes
   Sunny        Mild  Yes
Overcast        Mild  Yes
Overcast         Hot  Yes
    Rain        Mild   No
   Sunny         Hot   No
    Rain        Cool  Yes

=== Test Set Predictions ===
Sample 1 | {'Outlook': 'Overcast', 'Temperature': 'Mild'} | Actual: Yes | Predicted: Yes | Probs: {'No': 0.1627, 'Yes': 0.8373}
Sample 2 | {'Outlook': 'Overcast', 'Temperature': 'Hot'} | Actual: Yes | Predicted: Yes | Probs: {'No': 0.3683, 'Yes': 0.6317}
Sample 3 | {'Outlook': 'Rain', 'Temperature': 'Mild'} | Actual: No  | Predicted: Yes | Probs: {'No': 0.2257, 'Yes': 0.7743}
Sample 4 | {'Outlook': 'Sunny', 'Temperature': 'Hot'} | Actual: No  | Predicted: No  | Probs: {'No': 0.6999, 'Yes': 0.3001}
Sample 5